# Path B stress validation: QUIC wire throughput (pcap + tshark)

Compare **baseline** vs **qaccess_t_dynamic** from one Path B stress session (`pathB_stress` scenario + Fig.7 dynamic TBF on `h2-eth1`).

**Method**: per-run `pathA` / `pathB` pcaps, `tshark` with directional display filters, bin by second, Mbps = bytes×8/1e6. **Total QUIC wire throughput** = Path A downlink + Path B downlink (server → client).

Configured Path B capacity (server egress TBF): 0–50s @ 20 Mbps → 50–100s @ 30 Mbps → 100s+ @ 10 Mbps.

---

**Note:** Metric is **pcap-derived QUIC wire throughput** (headers and retransmissions included), not application goodput. Tshark uses `udp` in display filters; all tables and plot labels say **QUIC wire throughput**.


## 1. Setup

In [ ]:
import importlib.util
import subprocess
import sys

_PACKAGES = ["pandas", "matplotlib"]

def _have(mod: str) -> bool:
    return importlib.util.find_spec(mod) is not None

_missing = [p for p in _PACKAGES if not _have(p)]
if _missing:
    print("Installing:", ", ".join(_missing), flush=True)
    subprocess.check_call(
        [sys.executable, "-m", "pip", "install", "--disable-pip-version-check", "--no-input", *_missing],
        timeout=600,
    )
else:
    print("OK:", ", ".join(_PACKAGES))


In [ ]:
import math
import os
import shutil
import subprocess
from collections import defaultdict
from pathlib import Path

import matplotlib.pyplot as plt
import pandas as pd

try:
    from IPython import get_ipython
    _ip = get_ipython()
    if _ip is not None:
        _ip.run_line_magic("matplotlib", "inline")
except (ImportError, AttributeError):
    os.environ.setdefault("MPLBACKEND", "Agg")


def find_repo() -> Path:
    cwd = Path.cwd().resolve()
    for p in [cwd, *cwd.parents]:
        if (p / "scripts" / "analyze" / "parse_logs.py").is_file():
            return p
    return cwd


REPO = find_repo()
print("REPO =", REPO)


## 2. Run paths

In [ ]:
# Set SESSION to your logs_exp/session_pathB_stress_<timestamp> directory.
SESSION = REPO / "logs_exp" / "session_pathB_stress_<timestamp>"

RUNS = {
    "baseline": SESSION / "pathB_stress_baseline",
    "qaccess_t_dynamic": SESSION / "pathB_stress_qaccess_t_dynamic",
}

OUT_DIR = SESSION / "compare_csv"
OUT_DIR.mkdir(parents=True, exist_ok=True)

# Directional downlink filters (server → client)
PATH_A_DOWNLINK_FILTER = "udp && ip.src == 10.0.1.2 && ip.dst == 10.0.1.1"
PATH_B_DOWNLINK_FILTER = "udp && ip.src == 10.0.2.2 && ip.dst == 10.0.2.1"

SECOND_MAX = 220

WINDOWS = [
    ("0-50", 0, 50),
    ("50-60", 50, 60),
    ("50-100", 50, 100),
    ("100-110", 100, 110),
    ("100-150", 100, 150),
]

METHOD_LABELS = {
    "baseline": "baseline",
    "qaccess_t_dynamic": "qaccess_t_dynamic",
}

CSV_TIMESERIES = {
    "baseline": OUT_DIR / "throughput_quic_timeseries_baseline.csv",
    "qaccess_t_dynamic": OUT_DIR / "throughput_quic_timeseries_qaccess_t_dynamic.csv",
}
CSV_WINDOW_SUMMARY = OUT_DIR / "pathB_stress_quic_window_summary.csv"

FIG_TOTAL = OUT_DIR / "pathB_stress_total_quic_throughput.png"
FIG_PATHS = OUT_DIR / "pathB_stress_per_path_quic_throughput.png"
FIG_PATHB_VS_CAP = OUT_DIR / "pathB_stress_pathB_downlink_vs_capacity.png"


def configured_path_b_capacity_mbps(second: int) -> float:
    if second < 50:
        return 20.0
    if second < 100:
        return 30.0
    return 10.0


def find_pcaps(run_dir: Path):
    pcaps = sorted((run_dir / "pcaps").glob("path*.pcap"))
    pcap_a = next(p for p in pcaps if "pathA" in p.name)
    pcap_b = next(p for p in pcaps if "pathB" in p.name)
    return pcap_a, pcap_b


print("Session path:", SESSION.resolve())
assert shutil.which("tshark"), "tshark not on PATH (install Wireshark)"

for label, run_dir in RUNS.items():
    pcap_a, pcap_b = find_pcaps(run_dir)
    print(f"\n[{label}] run:", run_dir)
    print("  Path A pcap:", pcap_a, "exists" if pcap_a.is_file() else "MISSING")
    print("  Path B pcap:", pcap_b, "exists" if pcap_b.is_file() else "MISSING")


## 3. Pcap reader

In [ ]:
def read_frame_len_bins(pcap: Path, display_filter: str) -> defaultdict:
    """Bin frame bytes by second: floor(frame.time_relative)."""
    bins = defaultdict(int)
    cmd = [
        "tshark", "-r", str(pcap), "-Y", display_filter,
        "-T", "fields", "-E", "separator=\t",
        "-e", "frame.time_relative", "-e", "frame.len",
    ]
    proc = subprocess.run(cmd, stdout=subprocess.PIPE, stderr=subprocess.PIPE, text=True)
    if proc.returncode != 0:
        raise RuntimeError(proc.stderr.strip() or f"tshark failed: {pcap}")
    for line in proc.stdout.splitlines():
        parts = line.strip().split("\t")
        if len(parts) < 2:
            continue
        try:
            si = int(math.floor(float(parts[0])))
            flen = int(parts[1])
        except ValueError:
            continue
        if si >= 0:
            bins[si] += flen
    return bins


def bins_to_mbps_series(bin_dict: defaultdict, last_second: int, interval_s: float = 1.0) -> pd.Series:
    return pd.Series(
        [bin_dict.get(s, 0) * 8 / 1_000_000 / interval_s for s in range(0, last_second + 1)],
        dtype="float64",
    )


def mean_in_window(series: pd.Series, seconds: pd.Series, lo: int, hi: int) -> float:
    w = series[(seconds >= lo) & (seconds < hi)]
    return float(w.mean()) if len(w) else float("nan")


def max_in_window(series: pd.Series, seconds: pd.Series, lo: int, hi: int) -> float:
    w = series[(seconds >= lo) & (seconds < hi)]
    return float(w.max()) if len(w) else float("nan")


def improvement_pct(from_mbps: float, to_mbps: float) -> float:
    if from_mbps > 0 and math.isfinite(from_mbps) and math.isfinite(to_mbps):
        return (to_mbps - from_mbps) / from_mbps * 100.0
    return float("nan")


## 4. Build per-run timeseries

In [ ]:
def load_run_quic_timeseries(label: str, run_dir: Path) -> pd.DataFrame:
    pcap_a, pcap_b = find_pcaps(run_dir)
    print(f"Reading [{label}] pcaps for QUIC wire throughput...", flush=True)
    bins_a = read_frame_len_bins(pcap_a, PATH_A_DOWNLINK_FILTER)
    bins_b = read_frame_len_bins(pcap_b, PATH_B_DOWNLINK_FILTER)
    pcap_last = int(max(max(bins_a.keys(), default=0), max(bins_b.keys(), default=0)))
    last_s = int(min(SECOND_MAX, pcap_last))
    path_a = bins_to_mbps_series(bins_a, last_s).values
    path_b = bins_to_mbps_series(bins_b, last_s).values
    df = pd.DataFrame({
        "second": range(0, last_s + 1),
        "pathA_quic_mbps": path_a,
        "pathB_downlink_quic_mbps": path_b,
    })
    df["pathB_quic_mbps"] = df["pathB_downlink_quic_mbps"]
    df["total_quic_mbps"] = df["pathA_quic_mbps"] + df["pathB_downlink_quic_mbps"]
    out_path = CSV_TIMESERIES[label]
    df.to_csv(out_path, index=False)
    print(f"  Wrote QUIC wire throughput timeseries: {out_path} (rows={len(df)})")
    return df


run_dfs = {}
last_s = 0
for label, run_dir in RUNS.items():
    df = load_run_quic_timeseries(label, run_dir)
    run_dfs[label] = df
    last_s = max(last_s, int(df["second"].max()))

df_baseline = run_dfs["baseline"]
df_dynamic = run_dfs["qaccess_t_dynamic"]

print("\nPer-run timeseries CSV paths:")
for label, path in CSV_TIMESERIES.items():
    print(f"  {label}: {path}")
print("done. last_s =", last_s)


## 5. Window summary

In [ ]:
window_defs = WINDOWS + [("full", 0, last_s + 1)]

summary_rows = []
for wname, lo, hi in window_defs:
    b_total = mean_in_window(df_baseline["total_quic_mbps"], df_baseline["second"], lo, hi)
    d_total = mean_in_window(df_dynamic["total_quic_mbps"], df_dynamic["second"], lo, hi)
    b_pb = mean_in_window(df_baseline["pathB_downlink_quic_mbps"], df_baseline["second"], lo, hi)
    d_pb = mean_in_window(df_dynamic["pathB_downlink_quic_mbps"], df_dynamic["second"], lo, hi)
    b_share = (b_pb / b_total * 100.0) if b_total > 0 else float("nan")
    d_share = (d_pb / d_total * 100.0) if d_total > 0 else float("nan")
    summary_rows.append({
        "window": wname,
        "baseline_total_mbps": round(b_total, 3),
        "qaccess_t_dynamic_total_mbps": round(d_total, 3),
        "improvement_pct": round(improvement_pct(b_total, d_total), 2),
        "baseline_pathB_downlink_mbps": round(b_pb, 3),
        "qaccess_t_dynamic_pathB_downlink_mbps": round(d_pb, 3),
        "baseline_pathB_share_pct": round(b_share, 2),
        "qaccess_t_dynamic_pathB_share_pct": round(d_share, 2),
    })

df_window_summary = pd.DataFrame(summary_rows)
df_window_summary.to_csv(CSV_WINDOW_SUMMARY, index=False)

print("Window summary (mean QUIC wire throughput, Mbps):")
print("Wrote", CSV_WINDOW_SUMMARY)
display(df_window_summary)


## 6. Plots

In [ ]:
# 6a. Total QUIC wire throughput over time
fig, ax = plt.subplots(figsize=(12, 5))
ax.plot(df_baseline["second"], df_baseline["total_quic_mbps"], label="baseline", linewidth=1.2)
ax.plot(df_dynamic["second"], df_dynamic["total_quic_mbps"], label="qaccess_t_dynamic", linewidth=1.2)
for t in (50, 100):
    ax.axvline(t, linestyle="--", linewidth=1, color="gray", alpha=0.7)
ax.set_xlabel("Time from pcap start (s)")
ax.set_ylabel("QUIC wire throughput (Mbps)")
ax.set_title("Path B stress: Total QUIC Wire Throughput Over Time")
ax.legend(loc="best")
ax.grid(True, alpha=0.3)
fig.tight_layout()
fig.savefig(FIG_TOTAL, dpi=200, bbox_inches="tight")
plt.show()
print("Saved", FIG_TOTAL)


In [ ]:
# 6b. Per-path QUIC wire throughput (baseline vs qaccess_t_dynamic)
fig, axes = plt.subplots(2, 1, figsize=(12, 8), sharex=True)

for ax, col, title in [
    (axes[0], "pathA_quic_mbps", "Path A downlink QUIC wire throughput"),
    (axes[1], "pathB_downlink_quic_mbps", "Path B downlink QUIC wire throughput"),
]:
    ax.plot(df_baseline["second"], df_baseline[col], label="baseline", linewidth=1.2)
    ax.plot(df_dynamic["second"], df_dynamic[col], label="qaccess_t_dynamic", linewidth=1.2)
    for t in (50, 100):
        ax.axvline(t, linestyle="--", linewidth=1, color="gray", alpha=0.7)
    ax.set_ylabel("QUIC wire throughput (Mbps)")
    ax.set_title(title)
    ax.legend(loc="best")
    ax.grid(True, alpha=0.3)

axes[1].set_xlabel("Time from pcap start (s)")
fig.suptitle("Path B stress: Per-Path QUIC Wire Throughput", y=1.02)
fig.tight_layout()
fig.savefig(FIG_PATHS, dpi=200, bbox_inches="tight")
plt.show()
print("Saved", FIG_PATHS)


In [ ]:
# 6c. Path B downlink QUIC wire throughput vs configured capacity
seconds = df_baseline["second"].astype(int)
cap = [configured_path_b_capacity_mbps(int(s)) for s in seconds]

fig, ax = plt.subplots(figsize=(12, 5))
ax.plot(df_baseline["second"], df_baseline["pathB_downlink_quic_mbps"], label="baseline", linewidth=1.2)
ax.plot(df_dynamic["second"], df_dynamic["pathB_downlink_quic_mbps"], label="qaccess_t_dynamic", linewidth=1.2)
ax.plot(seconds, cap, label="configured Path B capacity", linewidth=1.5, linestyle="--", color="black")
for t in (50, 100):
    ax.axvline(t, linestyle=":", linewidth=1, color="gray", alpha=0.7)
ax.set_xlabel("Time from pcap start (s)")
ax.set_ylabel("QUIC wire throughput (Mbps)")
ax.set_title("Path B Downlink QUIC Wire Throughput vs Configured Capacity")
ax.legend(loc="best")
ax.grid(True, alpha=0.3)
fig.tight_layout()
fig.savefig(FIG_PATHB_VS_CAP, dpi=200, bbox_inches="tight")
plt.show()
print("Saved", FIG_PATHB_VS_CAP)


## 7. Path B utilization diagnosis

In [ ]:
def diagnose_path_b_utilization(df: pd.DataFrame, label: str) -> None:
    sec = df["second"]
    pb = df["pathB_downlink_quic_mbps"]
    mean_50_100 = mean_in_window(pb, sec, 50, 100)
    max_50_100 = max_in_window(pb, sec, 50, 100)
    mean_100_150 = mean_in_window(pb, sec, 100, 150)
    max_after_100 = max_in_window(pb, sec, 100, last_s + 1)
    after_100 = df[sec >= 100]
    n_above_10_5 = int((after_100["pathB_downlink_quic_mbps"] > 10.5).sum())

    print(f"\n--- Path B utilization ({label}) ---")
    print(f"  Path B downlink mean in 50–100s: {mean_50_100:.3f} Mbps")
    print(f"  Path B downlink max in 50–100s: {max_50_100:.3f} Mbps")
    print(f"  Path B downlink mean in 100–150s: {mean_100_150:.3f} Mbps")
    print(f"  Path B downlink max after 100s: {max_after_100:.3f} Mbps")
    print(f"  Seconds after 100s with Path B downlink > 10.5 Mbps: {n_above_10_5}")

    return mean_50_100, mean_100_150, max_after_100, n_above_10_5


print("Path B utilization diagnosis (QUIC wire throughput)")
b_mean_50_100, b_mean_100_150, b_max_after_100, b_n_high = diagnose_path_b_utilization(df_baseline, "baseline")
d_mean_50_100, d_mean_100_150, d_max_after_100, d_n_high = diagnose_path_b_utilization(df_dynamic, "qaccess_t_dynamic")

print("\n--- Interpretation (baseline) ---")
if b_mean_50_100 <= 10.0:
    print(
        "Path B is still under-utilized before the 100s capacity decrease. "
        "The 30→10 Mbps capacity change may not create a visible drop."
    )
elif b_mean_50_100 > 10.0 and b_mean_100_150 < 10.0:
    print(
        "The Path B stress scenario successfully exposes the configured 30→10 Mbps capacity decrease."
    )
else:
    print(
        "Mixed signal: Path B was used before 100s, but the post-100s mean did not fall clearly below 10 Mbps. "
        "Inspect the timeseries plot and per-second values."
    )

if b_max_after_100 > 10.5 or b_n_high > 0:
    print(
        "Warning: Path B downlink exceeds the configured post-100s capacity. "
        "Check tc direction, interface, or pcap time alignment."
    )


## 8. Summary

In [ ]:
print("=" * 60)
print("Path B stress validation — QUIC wire throughput summary")
print("=" * 60)
print("Session path:", SESSION.resolve())
print("Compares baseline vs qaccess_t_dynamic only (qaccess_t_static not used).")

print("\nPcap files per run:")
for label, run_dir in RUNS.items():
    pa, pb = find_pcaps(run_dir)
    print(f"  [{label}]")
    print(f"    Path A: {pa.name}")
    print(f"    Path B: {pb.name}")

print("\nPath B downlink filter:", PATH_B_DOWNLINK_FILTER)

print("\nOutput CSV paths:")
for label, path in CSV_TIMESERIES.items():
    print(f"  timeseries {label}: {path}")
print(f"  window summary: {CSV_WINDOW_SUMMARY}")

print("\nOutput figures:")
for p in (FIG_TOTAL, FIG_PATHS, FIG_PATHB_VS_CAP):
    print(f"  {p}")

print("\nWindow summary:")
display(df_window_summary)
